#### Importing required libraries 

In [50]:
from utils import Hetero_Data_Processor_Filter_on_Test_since_first_post # Required class for GNN Incremental training
import pandas as pd
from torch import nn
from torch_geometric.nn import GATConv,to_hetero
import torch.nn.functional as F
import numpy as np
import mlflow
import torch
import torch.nn as nn
import warnings
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
warnings.filterwarnings("ignore")

#### Testing a single load 

In [51]:
file_path_replies = r"../replies_sydneysiege.pkl"
file_path_posts = r"../posts_sydneysiege.pkl"


processor = Hetero_Data_Processor_Filter_on_Test_since_first_post(file_path_replies, file_path_posts, time_cut=1000)
data = processor.process()


In [25]:
data

HeteroData(
  id={
    x=[1010, 106],
    y=[1010],
    train_mask=[1010],
    val_mask=[1010],
    test_mask=[1010],
  },
  reply_user_id={ x=[11234, 104] },
  (id, retweet, reply_user_id)={ edge_index=[2, 11234] },
  (reply_user_id, rev_retweet, id)={ edge_index=[2, 11234] }
)

In [26]:
class GAT(torch.nn.Module):
    """
    Graph Attention Network (GAT) model with two GATConv layers and a final
    linear projection layer.

    This architecture applies graph attention mechanisms to learn contextual
    node embeddings based on graph connectivity. Dropout is applied after each
    GAT layer to help reduce overfitting.

    Parameters
    ----------
    dim_h : int
        Number of hidden attention heads/features in the first GAT layer.
    dim_i : int
        Number of intermediate output features of the second GAT layer.
    dim_out : int
        Output feature dimension, typically corresponding to the number of
        prediction classes or embedding size.

    Attributes
    ----------
    conv1 : GATConv
        First graph attention convolution layer with learned attention weights.
    conv2 : GATConv
        Second graph attention convolution layer that refines node embeddings.
    linear : nn.Linear
        Fully connected layer to project the learned embeddings to the output dimension.
    dropout : nn.Dropout
        Dropout layer applied after each convolution to reduce overfitting.

    Forward Inputs
    --------------
    x : torch.Tensor
        Node feature matrix of shape [num_nodes, num_features].
    edge_index : torch.LongTensor
        Graph edge index tensor of shape [2, num_edges] defining connectivity.

    Returns
    -------
    torch.Tensor
        Output node representations of shape [num_nodes, dim_out].
    """

    def __init__(self, dim_h,dim_i, dim_out):
        super().__init__()
        self.conv1 = GATConv((-1, -1), dim_h, add_self_loops=False)
        self.conv2 = GATConv(dim_h, dim_i, add_self_loops=False)
        self.linear = nn.Linear(dim_i, dim_out)
        self.dropout = nn.Dropout(p=0.4)

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index).relu()
        h = self.dropout(h)
        h = self.conv2(h, edge_index).relu()
        h = self.dropout(h)
        h = self.linear(h)
        return h


In [44]:


def evaluate(model, data, mask_names):


    """
    Evaluate a graph classification model using masked subsets of the data.

    This function runs the model in evaluation mode, computes predictions,
    filters them using one or multiple masks from the input dataset, and
    returns common binary classification metrics.

    Parameters
    ----------
    model : torch.nn.Module
        Trained GNN model producing class logits from graph inputs.
    data : torch_geometric.data.HeteroData
        Heterogeneous graph data structure containing:
        - `x_dict`: dictionary of node feature matrices
        - `edge_index_dict`: dictionary of edge connectivity
        - `'id'` node type with attributes `y` and boolean masks
          (e.g., 'train_mask', 'val_mask', 'test_mask')
    mask_names : str or list[str]
        Name(s) of mask attributes to evaluate on. If multiple masks
        are provided, they are combined using logical OR.

    Returns
    -------
    tuple(float, float, float, float)
        A tuple containing:
        - acc : float
            Accuracy score.
        - precision : float
            Proportion of predicted positives that are correctly classified.
        - recall : float
            True positive rate.
        - auc : float
            ROC-AUC score based on predicted class probabilities.

    Notes
    -----
    - Metrics are computed only on masked nodes.
    - If ROC-AUC cannot be computed due to a single class present in labels,
      a value of 0.0 is returned.
    """

    model.eval()
    out = model(data.x_dict, data.edge_index_dict)
    preds = out.argmax(dim=-1)
    labels = data['id'].y

    if isinstance(mask_names, str):
        mask = data['id'][mask_names]
    else:
        mask = torch.zeros_like(data['id'].y, dtype=torch.bool)
        for name in mask_names:
            mask |= data['id'][name]

    preds_masked = preds[mask]
    labels_masked = labels[mask]
    probs = out[mask][:, 1]  # Fixed: out is a tensor

    acc = accuracy_score(labels_masked.cpu(), preds_masked.cpu())
    precision = precision_score(labels_masked.cpu(), preds_masked.cpu(), zero_division=0)
    recall = recall_score(labels_masked.cpu(), preds_masked.cpu(), zero_division=0)

    try:
        auc = roc_auc_score(labels_masked.cpu(), probs.detach().cpu())
    except ValueError:
        auc = 0.0

    return acc, precision, recall, auc






#### Example  training

In [52]:

model = GAT(dim_h=64,dim_i=32, dim_out=2)
model = to_hetero(model, data.metadata(), aggr='sum')

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data, model = data.to(device), model.to(device)

In [53]:
for epoch in range(1, 101):
            model.train()
            optimizer.zero_grad()
            out = model(data.x_dict, data.edge_index_dict)['id']
            mask = data['id'].train_mask
            loss = F.cross_entropy(out[mask], data['id'].y[mask])
            loss.backward()
            optimizer.step()

            if epoch % 100 == 0:
                 train_acc, train_prec, train_recall, train_auc= evaluate(model, data, data['id'].train_mask)
                 print(f"[Epoch {epoch}] Train Loss: {loss:.4f} | Train Recall: {train_recall:.4f} | Train Auc: {train_auc:.4f}")

# Final evaluation on val + test
final_mask = data['id'].val_mask | data['id'].test_mask
acc, prec, recall, auc = evaluate_metrics(model, data, final_mask)
print(f"[Final Val+Test] Acc: {acc:.4f} | Prec: {prec:.4f} | Recall: {recall:.4f} | AUC: {auc:.4f}")

[Epoch 100] Train Loss: 0.4324 | Train Recall: 0.8429 | Train Auc: 0.9009
[Final Val+Test] Acc: 0.7244 | Prec: 0.7122 | Recall: 0.7432 | AUC: 0.7817


**Creating evaluate_metrics function to assess classification when new posts are created**

In [16]:
def evaluate_metrics(model, data, mask):


    """
    Compute evaluation metrics for a model on a masked node subset.

    The function performs prediction using the trained model, extracts
    predictions over a specific boolean mask, and computes standard
    classification metrics including accuracy, macro precision, macro
    recall, and ROC-AUC.

    Parameters
    ----------
    model : torch.nn.Module
        Trained GNN model that outputs logits for the `'id'` node type.
    data : torch_geometric.data.HeteroData
        Heterogeneous graph data containing node features, edge connections,
        labels, and a boolean mask to filter evaluation nodes.
    mask : torch.Tensor or list[bool]
        Boolean mask selecting the subset of nodes to evaluate.

    Returns
    -------
    tuple(float, float, float, float)
        A tuple of:
        - acc : float
            Accuracy score.
        - prec : float
            Macro-averaged precision.
        - recall : float
            Macro-averaged recall.
        - auc : float
            ROC-AUC score based on probability of the positive class.

    Notes
    -----
    - Evaluation is performed inside a `torch.no_grad()` block to disable gradient tracking.
    - If ROC-AUC computation fails (e.g., only one class present), a value of 0.0 is returned.
    """


    model.eval()
    with torch.no_grad():
        out = model(data.x_dict, data.edge_index_dict)['id']
        preds = out.argmax(dim=1)
        probs = out[:, 1]  # Probability of class 1

    true = data['id'].y[mask]
    pred = preds[mask]
    prob = probs[mask]

    acc = accuracy_score(true.cpu(), pred.cpu())
    prec = precision_score(true.cpu(), pred.cpu(), average='macro', zero_division=0)
    recall = recall_score(true.cpu(), pred.cpu(), average='macro', zero_division=0)
    try:
        auc = roc_auc_score(true.cpu(), prob.cpu())
    except:
        auc = 0.0

    return acc, prec, recall, auc

#### Setting MLflow Experiment

In [61]:

mlflow.set_experiment("GAT 2025-11-04 Sydney Siege")

2025/11/09 14:16:29 INFO mlflow.tracking.fluent: Experiment with name 'GAT 2025-11-04 Sydney Siege' does not exist. Creating a new experiment.


<Experiment: artifact_location='/workspaces/rumour-detection-gnn/New experiments/mlruns/88', creation_time=1762697789758, experiment_id='88', last_update_time=1762697789758, lifecycle_stage='active', name='GAT 2025-11-04 Sydney Siege', tags={}>

#### Loading dataset statistics to get the final time cut 

In [56]:
df_posts_by_tm = pd.read_csv('sydneysiege_posts_by_time_cut.csv')

df_posts_by_tm['new_posts_cum_sum'] = df_posts_by_tm.new_posts.cumsum()

max_time_cut = int(df_posts_by_tm[df_posts_by_tm.new_posts_cum_sum==int(df_posts_by_tm.new_posts_cum_sum.max())]\
                   .time_cut.min())

In [60]:
df_posts_by_tm.tail()

,Unnamed: 0,time_cut,rumours,post,previous_post,new_posts,no_rumours,perc_dataset,new_posts_cum_sum
4300,4300,4305,499,1172,1172.0,0.0,673,1.0,1171.0
4301,4301,4306,499,1172,1172.0,0.0,673,1.0,1171.0
4302,4302,4307,499,1172,1172.0,0.0,673,1.0,1171.0
4303,4303,4308,499,1172,1172.0,0.0,673,1.0,1171.0
4304,4304,4309,499,1172,1172.0,0.0,673,1.0,1171.0


* **The initial  time cut will be 10 minutes after the first post publication**
*  **The final time cut will be equal to 6 hours after the publication of last post**

In [63]:

previous_node_count = 0  # Start with no nodes

for time_cut in range(10, max_time_cut+(60*6), 10):
    print(f"\n=== Time Cut: {time_cut} minutes ===")

    processor = Hetero_Data_Processor_Filter_on_Test_since_first_post(file_path_replies, file_path_posts, time_cut=time_cut)
    data = processor.process()

    current_node_count = data['id'].x.shape[0]
    new_node_indices = np.arange(previous_node_count, current_node_count)
    previous_node_count = current_node_count

    # Set up model and training

    model = GAT(dim_h=64,dim_i=32, dim_out=2)
    model = to_hetero(model, data.metadata(), aggr='sum')
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data, model = data.to(device), model.to(device)
    
    with mlflow.start_run(run_name=f"time_cut_{time_cut}"):
        for epoch in range(1, 201):
            model.train()
            optimizer.zero_grad()
            out = model(data.x_dict, data.edge_index_dict)['id']
            mask = data['id'].train_mask
            loss = F.cross_entropy(out[mask], data['id'].y[mask])
            loss.backward()
            optimizer.step()

            if epoch % 100 == 0:
                 train_acc, train_prec, train_recall, train_auc= evaluate_metrics(model, data, data['id'].train_mask)
                 print(f"[Epoch {epoch}] Train Loss: {loss:.4f} | Train Recall: {train_recall:.4f} | Train Precision: {train_prec:.4f}")
    
        # Evaluate all predictions
        model.eval()
        with torch.no_grad():
            out = model(data.x_dict, data.edge_index_dict)['id']
            preds = out.argmax(dim=1)
            probs = out[:, 1]
    
        # New instances in val/test set
        val_test_mask = (data['id'].val_mask | data['id'].test_mask).cpu().numpy()
        new_instance_mask = np.zeros_like(val_test_mask, dtype=bool)
        new_instance_mask[new_node_indices] = True
        final_mask = new_instance_mask & val_test_mask
    
        if final_mask.sum() > 0:
            # Compute metrics
            true_new = data['id'].y.cpu().numpy()[final_mask]
            pred_new = preds.cpu().numpy()[final_mask]
            prob_new = probs.cpu().numpy()[final_mask]
        
            new_precision = precision_score(true_new, pred_new, average='macro', zero_division=0)
            new_recall = recall_score(true_new, pred_new, average='macro', zero_division=0)
            new_acc = accuracy_score(true_new, pred_new)
        else:
            new_precision = 0
            new_recall = 0
            new_acc =0
            print("No new instances to evaluate.")
            
    
        # Compute metrics
        
        all_eval_mask = data['id'].val_mask | data['id'].test_mask
        acc, prec, recall, auc = evaluate_metrics(model, data, all_eval_mask)
        
        print(f"[Final Val+Test] Acc: {acc:.4f} | Prec: {prec:.4f} | Recall: {recall:.4f} | AUC: {auc:.4f}")
    
    
        print(f"New Instances: {final_mask.sum()}")
        print(f"New Precision: {new_precision:.4f} | New Recall: {new_recall:.4f}")

        mlflow.log_metric("new_posts", final_mask.sum())
        
        mlflow.log_metric("final_precision", prec)
        mlflow.log_metric("final_recall", recall)
        mlflow.log_metric("final_auc", auc)
        mlflow.log_metric("final_acc", acc)

        mlflow.log_metric("curr_precision", new_precision)
        mlflow.log_metric("curr_recall", new_recall)
        mlflow.log_metric("curr_acc", new_acc)

        mlflow.log_metric("time_cut", time_cut)





=== Time Cut: 10 minutes ===
[Epoch 100] Train Loss: 0.5139 | Train Recall: 0.8342 | Train Precision: 0.8352
[Epoch 200] Train Loss: 0.3299 | Train Recall: 0.8817 | Train Precision: 0.8833
[Final Val+Test] Acc: 0.6667 | Prec: 0.4000 | Recall: 0.4000 | AUC: 0.6000
New Instances: 6
New Precision: 0.4000 | New Recall: 0.4000

=== Time Cut: 20 minutes ===
[Epoch 100] Train Loss: 0.4541 | Train Recall: 0.8424 | Train Precision: 0.8419
[Epoch 200] Train Loss: 0.2879 | Train Recall: 0.9032 | Train Precision: 0.9021
[Final Val+Test] Acc: 0.7143 | Prec: 0.6667 | Recall: 0.8333 | AUC: 0.9167
New Instances: 8
New Precision: 0.6667 | New Recall: 0.8571

=== Time Cut: 30 minutes ===
[Epoch 100] Train Loss: 0.4131 | Train Recall: 0.8567 | Train Precision: 0.8557
[Epoch 200] Train Loss: 0.2848 | Train Recall: 0.9102 | Train Precision: 0.9111
[Final Val+Test] Acc: 0.8636 | Prec: 0.6404 | Recall: 0.7000 | AUC: 0.9250
New Instances: 8
New Precision: 0.5000 | New Recall: 0.4375

=== Time Cut: 40 minutes

KeyboardInterrupt: 